In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_rows', None)

df_evasao = pd.read_csv("https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/main/Data/02_filtered/Maiores_Taxas_Evasao_e_Reprovacao_2024.csv")
df_enem = pd.read_csv("https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/main/Data/01_Cleaned/Tabela_ENEM_2024.csv")
df_censo = pd.read_csv("https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/main/Data/01_Cleaned/Tabela_Censo_Escolar_2024.csv", sep=";")

df_evasao['evasao_medio_total'] = pd.to_numeric(df_evasao['evasao_medio_total'].replace('Não informado', np.nan), errors='coerce')
df_enem['NOTA_GERAL'] = pd.to_numeric(df_enem['NOTA_GERAL'], errors='coerce')


df_escolas_censo = pd.merge(df_censo, df_evasao, left_on='CO_ENTIDADE', right_on='codigo_escola', how='inner')

evasao_state = df_escolas_censo.groupby('SG_UF')['evasao_medio_total'].mean().reset_index()
enem_state = df_enem.groupby('SG_UF_PROVA')['NOTA_GERAL'].mean().reset_index()


df_merged = pd.merge(evasao_state, enem_state, left_on='SG_UF', right_on='SG_UF_PROVA', how='inner')

df_merged = df_merged.sort_values(by='evasao_medio_total', ascending=False)

# ==========================================
# --- O PRINT DO RANKING COMPLETO ---
# ==========================================
print("--- RANKING DE EVASÃO NO ENSINO MÉDIO NAS ESCOLAS DO CENSO E NOTAS DO ENEM ---")

df_merged.index = np.arange(1, len(df_merged) + 1)
display(df_merged[['SG_UF', 'evasao_medio_total', 'NOTA_GERAL']].round(2))
print("\n")

corr_value = df_merged['evasao_medio_total'].corr(df_merged['NOTA_GERAL'])

plt.figure(figsize=(10, 6))
sns.regplot(data=df_merged, x='evasao_medio_total', y='NOTA_GERAL', color='b', scatter_kws={'s':50})

for i, row in df_merged.iterrows():
    plt.text(row['evasao_medio_total'], row['NOTA_GERAL'] + 1.5, row['SG_UF'], fontsize=9, ha='center')

media_nacional = df_enem['NOTA_GERAL'].mean()
plt.axhline(media_nacional, color='red', linestyle='--', label=f'Média Nacional ENEM ({media_nacional:.2f})')

plt.title(f'Correlação entre Evasão no Ensino Médio e Nota Geral do ENEM por Estado\n(Escolas do Censo Inep)')
plt.xlabel('Taxa Média de Evasão no Ensino Médio (%)')
plt.ylabel('Nota Média Geral no ENEM')
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()

plt.savefig('correlacao_evasao_enem.png')
plt.show()